# 70 — Train BGE-M3 bi-encoder (Stage A) — end-to-end

Single-path notebook. Fine-tunes BGE-M3 on a fraction of HF train,
trains on a user-disjoint train/val split, pushes the merged model
to HF Hub, re-embeds the ~47K-track catalog, and reports standalone
dev nDCG@20.

## Quick start

Set `SKIP_MINING = True` (default) in cell 1 to reuse an existing
`TRIPLES_JSONL` and run cells 2-7 end-to-end. Set `SKIP_MINING = False`
to force a fresh HN mine.


1. Set the parameters in **cell 1 (CONFIG)** — `MAX_INPUT_ROWS`, `EPOCHS`,
   `PER_DEVICE_BATCH_SIZE`, etc. — to control the run size.
2. Run cells 2-7 sequentially. Open cell 5 (TensorBoard) in parallel with cell 4 (train).

## What we're verifying

1. **train/val nDCG alignment** — `train/ndcg_inbatch` and `val/ndcg`
   curves should track each other (gap < ~0.05 sustained). Gap > 0.05 = leak.
2. **honest retrieval lift** — `val/full_catalog_ndcg_at_20` (vs the real ~47K
   catalog) should climb above ~0.05-0.08 by end of epoch 1.
3. **HF train/test disjointness** — preflight in cell 7 refuses to score
   if HF train and test splits share user_ids or session_ids.

## §6.5 amendment knobs active in this run

- `--split-key user_id` — user-disjoint train/val (every session of a given user lives in one partition).
- `UserDisjointBatchSampler` — distinct users per batch; without-replacement; heap-greedy load-balanced.
- In-batch InfoNCE with false-positive collision masking (same-pos-tid + cross-pos-into-neg-slot; RocketQAv2 / BGE-M3 §3.3).
- Per-row `K_data = N_NEGATIVES` mined negs; rows below threshold are dropped (no upsampling).


In [ ]:
# 1) CONFIG — all run parameters in one place. Edit then run cells 2-7.

# --- Branch + identity --------------------------------------------------
BRANCH       = 'fresh-model'
HUB_USER     = 'OrRim123'
RUN_NAME     = 'bge-m3-music-v1'  # used as suffix everywhere; matches production wRRF factory

# --- Data scope ---------------------------------------------------------
SKIP_MINING    = True           # True = use existing TRIPLES_JSONL if present;
                                # False = always re-mine.
MAX_INPUT_ROWS = 0              # 0 = use ALL ~121K HF train turns; set 12000 for quick-iter (~10%)
DEV_EVAL_ROWS  = 8000           # # of HF test rows scored in cell 7 (use full HF test split)

# --- HN mining ----------------------------------------------------------
PERCPOS_THRESHOLD = 0.95        # candidates KEPT if score < threshold * pos_score
POOL_SIZE         = 1000        # top-K candidates considered per query
N_NEGATIVES       = 15          # negs per row (rows below this are DROPPED at train time)
MINING_BATCH_SIZE = 64

# --- Training -----------------------------------------------------------
EPOCHS                     = 3        # 3 for production; 1 for quick-iter sanity
PER_DEVICE_BATCH_SIZE      = 16
GRAD_ACCUM_STEPS           = 2        # effective batch 32 (spec §6); set 1 for quick-iter
LR                         = 5e-6
TEMPERATURE                = 0.05
LORA_RANK                  = 64
LORA_ALPHA                 = 128
QUERY_MAX_LEN              = 384
PASSAGE_MAX_LEN            = 192
SPLIT_KEY                  = 'user_id'   # 'user_id' | 'session_id' | 'row'
VAL_FRACTION               = 0.10
LOGGING_STEPS              = 25       # log every N opt-steps; 5 for quick-iter
VAL_EVERY_N_STEPS          = 100      # val pass every N opt-steps; 10 for quick-iter
VAL_FULL_CATALOG_EVERY_N   = 200      # full-catalog (~50 sec each) every N opt-steps; 50 for quick-iter
CHECKPOINT_EVERY_N_EPOCHS  = 1        # save adapter per epoch (~30 MB each) for warm-start/revert
GRADIENT_CHECKPOINTING     = True
TENSORBOARD_PORT           = 6006

# --- Derived paths (do not edit usually) --------------------------------
HUB_REPO         = f'{HUB_USER}/recsys2026-{RUN_NAME}'
HUB_REPO_MERGED  = f'{HUB_REPO}-merged'
EMBED_LABEL      = f'{RUN_NAME}-merged'
TRIPLES_JSONL    = f'experiments/cache/retrieval_v2/triples_{RUN_NAME.replace("-","_")}.jsonl'
TRAIN_OUTPUT_DIR = f'/content/{RUN_NAME.replace("-","_")}'
DRIVE_RESULTS    = f'/content/drive/MyDrive/recsys2026_retrieval_v2_cache/results/{RUN_NAME.replace("-","_")}'
DRIVE_LOG_HN     = f'/content/drive/MyDrive/recsys2026_retrieval_v2_cache/{RUN_NAME.replace("-","_")}_hn_log.txt'
DRIVE_LOG_TRAIN  = f'/content/drive/MyDrive/recsys2026_retrieval_v2_cache/{RUN_NAME.replace("-","_")}_train_log.txt'
CACHE_ROOT       = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache/dense_local'
SAFE_MODEL       = HUB_REPO_MERGED.replace('/', '_')
CATALOG_OUT_DIR  = f'{CACHE_ROOT}/{SAFE_MODEL}/{EMBED_LABEL}'

print('Config loaded:')
for k in ('BRANCH','HUB_REPO','HUB_REPO_MERGED','TRIPLES_JSONL','TRAIN_OUTPUT_DIR',
         'SKIP_MINING','MAX_INPUT_ROWS','EPOCHS','PER_DEVICE_BATCH_SIZE','GRAD_ACCUM_STEPS',
         'N_NEGATIVES','SPLIT_KEY','VAL_FRACTION'):
    print(f'  {k} = {globals()[k]!r}')


In [ ]:
# 2) Setup — clone branch + HF auth + Drive mount + deps.
import os
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
for name, drive_subdir in [
    ('sid', 'recsys2026_sid_cache'),
    ('retrieval_v2', 'recsys2026_retrieval_v2_cache'),
]:
    src = f'{DRIVE_BASE}/{drive_subdir}'
    dst = f'{LOCAL_BASE}/{name}'
    os.makedirs(src, exist_ok=True)
    if os.path.islink(dst): os.unlink(dst)
    elif os.path.exists(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(src, dst)

!pip install -q --upgrade \
    'peft>=0.10' 'transformers>=4.40' 'accelerate>=0.30' \
    'sentence-transformers>=3.0' 'FlagEmbedding>=1.3' 'bm25s' 'torchao>=0.16' \
    'datasets' 'pandas<3.0' 'tqdm' 'omegaconf' 'pyyaml' 'tensorboard'


In [ ]:
# 3) HN mine — produces TRIPLES_JSONL from HF train.
# Honors SKIP_MINING: if the JSONL already exists on Drive and the flag
# is True, this cell skips the ~25-85 min mine and just confirms the
# file's there. Set SKIP_MINING = False in cell 1 to force a re-mine.
import os
RESULTS_DIR = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)

_existing_triples = os.path.exists(TRIPLES_JSONL)
if SKIP_MINING and _existing_triples:
    print(f'[mine] SKIP_MINING=True and {TRIPLES_JSONL} exists — skipping mine.')
    !wc -l {TRIPLES_JSONL}
    # Schema sanity even on the skipped path.
    !head -1 {TRIPLES_JSONL} | python3 -c "import json,sys; r=json.loads(sys.stdin.read()); print('HAS session_id:', 'session_id' in r); print('HAS user_id:', 'user_id' in r); print('HAS neg_tids:', 'neg_tids' in r); print('keys:', sorted(r.keys()))"
else:
    if SKIP_MINING and not _existing_triples:
        print(f'[mine] SKIP_MINING=True but {TRIPLES_JSONL} not found — mining anyway.')
    !rm -f {TRIPLES_JSONL}
    _max_rows_flag = f'--max-rows {MAX_INPUT_ROWS}' if MAX_INPUT_ROWS > 0 else ''
    !cd /content/recsys2026 && python -u scripts/build_bi_encoder_training_data.py \
        --train-conv-hf talkpl-ai/TalkPlayData-Challenge-Dataset \
        --output {TRIPLES_JSONL} \
        --query-mode bge_m3_structured \
        --percpos-threshold {PERCPOS_THRESHOLD} --pool-size {POOL_SIZE} \
        --k-negs {N_NEGATIVES} --batch-size {MINING_BATCH_SIZE} \
        {_max_rows_flag} \
        2>&1 | tee {DRIVE_LOG_HN}
    !wc -l {TRIPLES_JSONL}
    !head -1 {TRIPLES_JSONL} | python3 -c "import json,sys; r=json.loads(sys.stdin.read()); print('HAS session_id:', 'session_id' in r); print('HAS user_id:', 'user_id' in r); print('HAS neg_tids:', 'neg_tids' in r); print('keys:', sorted(r.keys()))"


In [ ]:
# 3b) (One-shot) Expand music-turn IDs in [HISTORY]: blocks to match
# production inference (id_to_metadata format). Closes the train/eval
# parity gap on the [HISTORY] music turns.
#
# This is idempotent and FAST (~1-2 min CPU): only the `query` field is
# rewritten; all other fields (pos/neg/pos_tid/neg_tids/user_id/session_id)
# are preserved byte-for-byte.
#
# After the builder fix in commit (see history-corpus-types arg), fresh
# mines from cell 3 already emit the corrected format → this cell becomes
# a no-op (rewrites 0 rows). Kept for safety + to salvage any pre-fix
# JSONL still around on Drive.
TRIPLES_FIXED = TRIPLES_JSONL.replace('.jsonl', '_history_fixed.jsonl')
!cd /content/recsys2026 && python -u scripts/expand_history_in_triples.py \
    --input {TRIPLES_JSONL} \
    --output {TRIPLES_FIXED} \
    --track-meta-hf talkpl-ai/TalkPlayData-Challenge-Track-Metadata \
    --corpus-types track_name,artist_name,album_name
# Swap in the fixed file so cell 4 (train) picks it up automatically.
!mv {TRIPLES_FIXED} {TRIPLES_JSONL}
# Confirm one row's [HISTORY] now contains `A: track_id: <id>, track_name:...`.
!python3 -c "import json; r=json.loads(open('{TRIPLES_JSONL}').readline()); print(r['query'][:600])"


In [ ]:
# 4) Train + merge LoRA into base + push to Hub.
# Hub target: <HUB_REPO_MERGED> (gets overwritten each run).
# Open cell 5 (TensorBoard) in PARALLEL after this cell starts.
_grad_ckpt_flag = '--gradient-checkpointing' if GRADIENT_CHECKPOINTING else '--no-gradient-checkpointing'
_ckpt_flag = f'--checkpoint-every-n-epochs {CHECKPOINT_EVERY_N_EPOCHS}'
!cd /content/recsys2026 && python -u scripts/train_bi_encoder.py \
    --triples {TRIPLES_JSONL} \
    --output-dir {TRAIN_OUTPUT_DIR} \
    --hub-repo {HUB_REPO} \
    --results-dir {DRIVE_RESULTS} \
    --epochs {EPOCHS} \
    --per-device-batch-size {PER_DEVICE_BATCH_SIZE} \
    --gradient-accumulation-steps {GRAD_ACCUM_STEPS} \
    --lr {LR} \
    --temperature {TEMPERATURE} \
    --lora-rank {LORA_RANK} \
    --lora-alpha {LORA_ALPHA} \
    --split-key {SPLIT_KEY} \
    --val-fraction {VAL_FRACTION} \
    --query-max-len {QUERY_MAX_LEN} \
    --passage-max-len {PASSAGE_MAX_LEN} \
    --logging-steps {LOGGING_STEPS} \
    --val-every-n-steps {VAL_EVERY_N_STEPS} \
    --val-full-catalog-every-n-steps {VAL_FULL_CATALOG_EVERY_N} \
    {_ckpt_flag} \
    {_grad_ckpt_flag} \
    --merge --cleanup-after-push \
    2>&1 | tee {DRIVE_LOG_TRAIN}


In [ ]:
# 5) TensorBoard launcher — open in PARALLEL with cell 4.
# Compare for the leak check:
#   train/ndcg_inbatch  ← per-batch on training rows
#   val/ndcg            ← per-batch on held-out USER-DISJOINT val users
# Aligned curves (gap < ~0.05 sustained) → no leak.
# Also watch val/full_catalog_ndcg_at_20 — should climb above ~0.05.
%load_ext tensorboard
%tensorboard --logdir {TRAIN_OUTPUT_DIR}/runs --port={TENSORBOARD_PORT}


In [ ]:
# 6) Re-embed the ~47K-track catalog with the fine-tuned (merged) model.
# EVAL ALIGNMENT: max_seq_length=PASSAGE_MAX_LEN + right-trunc mirror training.
import os, pickle, numpy as np, sys
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
from mcrs.retrieval_modules.bge_m3_format import format_track_text

os.makedirs(CATALOG_OUT_DIR, exist_ok=True)
model = SentenceTransformer(HUB_REPO_MERGED, device='cuda')
model.max_seq_length = PASSAGE_MAX_LEN
model.tokenizer.truncation_side = 'right'
print(f'[catalog re-embed] max_seq_length={model.max_seq_length} truncation_side={model.tokenizer.truncation_side}')

tm = load_dataset('talkpl-ai/TalkPlayData-Challenge-Track-Metadata', split='all_tracks')
texts = [format_track_text(r.get('track_name','unknown'), r.get('artist_name'), r.get('album_name'), r.get('release_date'), r.get('tag_list')) for r in tm]
track_ids = [r['track_id'] for r in tm]
embs = model.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
embs = np.asarray(embs, dtype=np.float32)
out_path = os.path.join(CATALOG_OUT_DIR, 'track_embeddings.pkl')
with open(out_path, 'wb') as f:
    pickle.dump({'track_ids': track_ids, 'track_mat': embs}, f)
print(f'wrote {len(track_ids)} embeddings → {out_path}')


In [ ]:
# 7) Dev eval — preflight (HF train↔test must be USER+SESSION disjoint)
# + standalone nDCG@20 on DEV_EVAL_ROWS rows from HF test split.
import sys, math, os, pickle
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
sys.path.insert(0, '/content/recsys2026/scripts')
from mcrs.retrieval_modules.bge_m3_format import format_query_text
from build_bi_encoder_training_data import _iter_conversation_turns

# Preflight: HF train↔test disjointness.
train_sess = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='train')
test_sess  = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')
train_users = {s.get('user_id') for s in train_sess if s.get('user_id')}
test_users  = {s.get('user_id') for s in test_sess  if s.get('user_id')}
train_sids  = {s.get('session_id') for s in train_sess if s.get('session_id')}
test_sids   = {s.get('session_id') for s in test_sess  if s.get('session_id')}
user_overlap = train_users & test_users
sess_overlap = train_sids & test_sids
print(f'[preflight] HF train users={len(train_users):,} test users={len(test_users):,} | overlap={len(user_overlap)}')
print(f'[preflight] HF train sessions={len(train_sids):,} test sessions={len(test_sids):,} | overlap={len(sess_overlap)}')
if user_overlap or sess_overlap:
    raise AssertionError(
        f'HF train/test splits are NOT disjoint: {len(user_overlap)} shared users, '
        f'{len(sess_overlap)} shared sessions. Refusing to score.'
    )
print('[preflight] HF train/test disjointness OK')

# Load fine-tuned model + re-embedded catalog.
catalog_pkl = os.path.join(CATALOG_OUT_DIR, 'track_embeddings.pkl')
with open(catalog_pkl, 'rb') as f:
    payload = pickle.load(f)
track_ids = payload['track_ids']
track_mat = payload['track_mat']

model = SentenceTransformer(HUB_REPO_MERGED, device='cuda')
model.max_seq_length = QUERY_MAX_LEN
model.tokenizer.truncation_side = 'left'
print(f'[dev eval] max_seq_length={model.max_seq_length} truncation_side={model.tokenizer.truncation_side}')

rows = _iter_conversation_turns(test_sess)[:DEV_EVAL_ROWS]
queries = [format_query_text(
    chat_history=r.get('chat_history') or [],
    current_user_query=r.get('current_user_query',''),
    user_profile=r.get('user_profile_raw'),
    conversation_goal=r.get('conversation_goal'),
    mode='bge_m3_structured',
) for r in rows]
q_emb = model.encode(queries, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
q_emb = np.asarray(q_emb, dtype=np.float32)
sims = q_emb @ track_mat.T
top20_idx = np.argpartition(-sims, kth=19, axis=1)[:, :20]
ri = np.arange(sims.shape[0])[:, None]
top20_sorted = top20_idx[ri, np.argsort(-sims[ri, top20_idx], axis=1)]
ndcgs = []
tid_to_idx = {tid: i for i, tid in enumerate(track_ids)}
for i, r in enumerate(rows):
    gold = r['track_id']
    if gold not in tid_to_idx:
        ndcgs.append(0.0); continue
    gold_idx = tid_to_idx[gold]
    top20_tids = top20_sorted[i]
    if gold_idx in top20_tids:
        rank = list(top20_tids).index(gold_idx) + 1
        ndcgs.append(1.0 / math.log2(rank + 1))
    else:
        ndcgs.append(0.0)
mean_ndcg = float(sum(ndcgs) / len(ndcgs))
print(f'\nfine-tuned BGE-M3 ({RUN_NAME}, {EPOCHS} epoch) standalone nDCG@20 on dev: {mean_ndcg:.4f}')
print(f'compare against val/ndcg in TB; gap should be small if model + data are healthy.')
